# Fleet maintenance cost: v24-v25

**v24: v9 ranking with risk x9; v25: risk x12**

Public leaderboard: not submitted

This notebook contains all of this version's code. Each script is written to disk with `%%writefile` (so scripts can import each other) and then run with `%run`.

**Before running:** put the competition files `train.csv`, `test.csv` and `sample_submission.csv` in a `data/` folder next to this notebook, and install `numpy pandas scipy scikit-learn lightgbm xgboost catboost`.

In [ ]:
import os
os.makedirs("oof", exist_ok=True)   # saved model predictions go here

## Shared modules

### `fleet_prep.py`
Loads the data and undoes the test-depot sensor shifts.

In [ ]:
%%writefile fleet_prep.py
"""Data loading and depot-shift corrections for the Fleet Maintenance competition.

Findings from adversarial validation (train vs test AUC was 1.000 before fixes):
  * s054..s063 look like a 10-bin histogram. In test the bins are shifted one
    column to the right (test s054 is always 0 and test s055..s063 match train
    s054..s062). We shift them back and mark test s063 as missing.
  * s120 has a constant offset of ~16,025,000 in test.
  * s122 is scaled by ~2.54 in test (looks like a unit change).
  * s047 is entirely missing in test, so it is dropped.
"""
import numpy as np
import pandas as pd

HIST = [f"s{i:03d}" for i in range(54, 64)]
S120_OFFSET = 16_025_000.0
S122_SCALE = 2.54


def fix_test(te: pd.DataFrame) -> pd.DataFrame:
    te = te.copy()
    vals = {f"s{i:03d}": te[f"s{i+1:03d}"].to_numpy() for i in range(54, 63)}
    for c, v in vals.items():
        te[c] = v
    te["s063"] = np.nan
    te["s120"] = np.maximum(te["s120"] - S120_OFFSET, 0)
    te["s122"] = te["s122"] / S122_SCALE
    return te


ROUNDING = {"s055": 10, "s062": 10, "s057": 100}  # test stores these coarser


def fix_train(tr: pd.DataFrame) -> pd.DataFrame:
    tr = tr.copy()
    for c, g in ROUNDING.items():
        tr[c] = np.round(tr[c] / g) * g
    return tr


def add_features(df: pd.DataFrame, sensor_cols) -> pd.DataFrame:
    out = df[sensor_cols].copy()
    out["n_missing"] = df[sensor_cols].isna().sum(1)
    out["n_zero"] = (df[sensor_cols] == 0).sum(1)
    h = df[HIST[:-1]]  # s063 is unavailable in test
    tot = h.sum(1, min_count=1)
    out["hist_total"] = tot
    for c in HIST[:-1]:
        out[f"{c}_frac"] = df[c] / tot.replace(0, np.nan)
    w = np.arange(len(HIST) - 1)
    out["hist_mean_bin"] = (h.fillna(0).to_numpy() * w).sum(1) / tot.replace(0, np.nan)
    return out


def load(data_dir="data"):
    tr = fix_train(pd.read_csv(f"{data_dir}/train.csv"))
    te = fix_test(pd.read_csv(f"{data_dir}/test.csv"))
    sensors = [c for c in te.columns if c.startswith("s") and c not in ("s047", "s063")]
    X = add_features(tr, sensors)
    Xt = add_features(te, sensors)
    return tr, te, X, Xt

### `policy.py`
Turns failure probability + repair-cost distribution into the cost-optimal inspect/budget decision.

In [ ]:
%%writefile policy.py
"""Cost-optimal inspect/budget decisions from probabilities and cost predictions.

Cost per vehicle (competition statement):
  not inspected: C (actual repair cost, 0 if healthy)
  inspected:     10 + 0.20 * max(C - b, 0) + 0.02 * max(b - C, 0)

For a vehicle with failure probability p and conditional repair-cost distribution
C | fail ~ exp(mu + eps) (eps = empirical out-of-fold residuals), the expected
inspection cost is minimised by the 0.20/0.22 = 0.909 quantile of the mixture
(0 with prob 1-p, C|fail with prob p). We inspect only when that expected cost is
below the expected cost of not inspecting, p * E[C | fail].
"""
import numpy as np

UNDER, OVER, FIXED = 0.20, 0.02, 10.0
TAU = UNDER / (UNDER + OVER)


def competition_cost(c, flag, budget):
    return np.where(flag == 1,
                    FIXED + UNDER * np.maximum(c - budget, 0) + OVER * np.maximum(budget - c, 0),
                    c)


def decide(p, mu, eps, alpha=1.0, eps_scale=1.0, margin=0.0, budget_mult=1.0, sigma=None):
    """Return flag, budget arrays.

    alpha       multiplies probabilities (calibration / prior adjustment)
    eps_scale   widens or narrows the residual distribution
    margin      extra cost an inspection must save before we flag
    budget_mult scales the optimal budget (kept at 1 unless CV says otherwise)
    sigma       optional per-vehicle residual scale (heteroscedastic cost model);
                then eps should be standardised residuals z = residual / sigma
    """
    sig = np.ones(len(p)) if sigma is None else np.asarray(sigma)
    p = np.clip(alpha * p, 1e-9, 1 - 1e-9)
    e = np.sort(eps * eps_scale)
    # samples of C|fail per vehicle: shape (n, k)
    out_flag = np.zeros(len(p), dtype=int)
    out_b = np.zeros(len(p))
    for s in range(0, len(p), 4000):
        sl = slice(s, s + 4000)
        C = np.exp(mu[sl, None] + sig[sl, None] * e[None, :])
        k = C.shape[1]
        ps = p[sl]
        u = 1 - (1 - TAU) / ps  # required quantile of C|fail
        idx = np.clip(np.ceil(u * k).astype(int) - 1, 0, k - 1)
        b = np.where(u > 0, C[np.arange(C.shape[0]), idx], 0.0) * budget_mult
        loss_fail = (UNDER * np.maximum(C - b[:, None], 0) + OVER * np.maximum(b[:, None] - C, 0)).mean(1)
        insp = FIXED + ps * loss_fail + (1 - ps) * OVER * b
        no_insp = ps * C.mean(1)
        f = (no_insp - insp > margin).astype(int)
        out_flag[sl] = f
        out_b[sl] = np.where(f == 1, b, 0.0)
    return out_flag, out_b

### `cv_eval.py`
Labels, repair costs and censoring weights shared by all scripts.

In [ ]:
%%writefile cv_eval.py
"""Shared helpers for comparing model versions on out-of-fold (OOF) predictions.

Each version is judged two ways:
  * OOF competition cost on the training depot (policy alpha tuned on OOF)
  * AP / log-loss of the failure probabilities
For test, every version is anchored to the same number of inspections as the
best leaderboard-validated setting (v10: 1754), so a new version changes *which*
vehicles are inspected, not *how many*.
"""
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from policy import decide, competition_cost

TARGET_INSPECTIONS = 1754

_tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost", "followup_days"])
y = _tr.failure.to_numpy()
cost = _tr.repair_cost.fillna(0).to_numpy()
weights = np.where((y == 0) & (_tr.followup_days < 365), _tr.followup_days / 365.0, 1.0)


def oof_cost(p, mu, eps, alphas=(1.0, 1.25, 1.5, 2.0), margins=(0, 1, 2)):
    best = min((competition_cost(cost, *decide(p, mu, eps, a, 1.0, m)).mean(), a, m)
               for a in alphas for m in margins)
    return best


def report(name, p, mu, eps):
    c, a, m = oof_cost(p, mu, eps)
    print(f"{name:28s} AUC {roc_auc_score(y, p):.4f}  AP {average_precision_score(y, p):.4f}  "
          f"LL {log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)):.5f}  OOF cost {c:.4f} (alpha {a}, margin {m})",
          flush=True)
    return c


def test_submission(pt, mut, eps, path, budget_mult=1.0, target=TARGET_INSPECTIONS):
    """Pick alpha so the test inspection count matches the leaderboard-validated target."""
    lo, hi = 0.1, 200.0
    for _ in range(40):
        mid = np.sqrt(lo * hi)
        n = decide(pt, mut, eps, mid, 1.0, 0.0, budget_mult)[0].sum()
        lo, hi = (mid, hi) if n < target else (lo, mid)
    f, b = decide(pt, mut, eps, hi, 1.0, 0.0, budget_mult)
    te = pd.read_csv("data/test.csv", usecols=["id"])
    sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
    assert len(sub) == 20000 and sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all()
    sub.to_csv(path, index=False)
    print(f"wrote {path}: alpha {hi:.2f}, inspections {int(f.sum())}")
    return sub

### `make_v12_v14.py`
v12-v14 builder (also imported by make_grid.py).

In [ ]:
%%writefile make_v12_v14.py
"""v12-v14: label-free prior-shift correction + better budgets + more bagging.

v12  EM prior-shift (Saerens et al., 2002): estimate the test depot's failure rate
     from unlabelled test predictions and adjust probabilities in odds space.
     Policy settings (alpha 1.5) are the ones tuned on training OOF cost; no
     leaderboard tuning. EM says the test failure rate is ~4.4x training.
v13  v12 inspections, budgets from a heteroscedastic cost model (per-vehicle
     spread of log repair cost, sigma_model.py). Same flags, -0.8..-2.1% budget
     cost on inspected vehicles in CV.
v14  v13 with test probabilities averaged over the 15 CV fold models per
     algorithm plus the 6 full-data models (more bagging, lower variance).
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost, weights

ALPHA, MARGIN = 1.5, 0.0
te = pd.read_csv("data/test.csv", usecols=["id"])
mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
mut = np.load("oof/full_mut.npy")
sig_t = np.load("oof/sigma_test.npy"); z = np.load("oof/z.npy")
ps = np.average(y, weights=weights)


def em_adjust(pt, it=500):
    pi = ps
    for _ in range(it):
        r = (pi / ps) * pt
        q = r / (r + ((1 - pi) / (1 - ps)) * (1 - pt))
        pi = q.mean()
    return q, pi


def write(name, flags, budgets):
    sub = pd.DataFrame({"id": te.id, "flag": flags.astype(int), "budget": np.round(budgets, 2)})
    assert len(sub) == 20000 and sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all()
    assert (sub.loc[sub.flag == 0, "budget"] == 0).all()
    sub.to_csv(f"submission_{name}.csv", index=False)
    print(f"{name}: inspections {int(flags.sum())}, mean budget {budgets[flags == 1].mean():.0f}")


def hetero_budgets(q, flags):
    # force a budget for every vehicle, then keep it only where flagged
    _, b = decide(q, mut, z, ALPHA, 1.0, -1e9, sigma=sig_t)
    return np.where(flags == 1, b, 0.0)


# v12
q12, pi = em_adjust(np.load("oof/full_pt.npy"))
print(f"EM test failure rate {pi:.4f} ({pi / ps:.1f}x train)")
f12, b12 = decide(q12, mut, eps, ALPHA, 1.0, MARGIN)
write("v12_em_prior", f12, b12)

# v13
write("v13_em_hetero_budget", f12, hetero_budgets(q12, f12))

# v14
cv_pt = np.mean([np.load(f"oof/pt_{n}_w.npy") for n in ("lgb", "xgb", "cat")], 0)
pt14 = (15 * cv_pt + 2 * np.load("oof/full_pt.npy")) / 17   # 45 CV models + 6 full-data models
q14, pi14 = em_adjust(pt14)
print(f"v14 EM test failure rate {pi14:.4f}")
f14, _ = decide(q14, mut, eps, ALPHA, 1.0, MARGIN)
write("v14_em_hetero_bagged", f14, hetero_budgets(q14, f14))

## Pipeline (run in order)

### Step 1: `oof_models.py`
Out-of-fold failure probabilities and conditional repair-cost predictions.

In [ ]:
%%writefile oof_models.py
"""Out-of-fold failure probabilities and conditional repair-cost predictions.

Writes oof/*.npy so the decision policy can be tuned without retraining.
"""
import os, sys, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
from fleet_prep import load

SEEDS = [0, 1, 2]
NF = 5
os.makedirs("oof", exist_ok=True)
tr, te, X, Xt = load()
y = tr.failure.to_numpy()
cost = tr.repair_cost.fillna(0).to_numpy()
# Censoring: healthy vehicles observed < 365 days are only partly informative.
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)


def clf_models(seed):
    return {
        "lgb": lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15,
                                  min_child_samples=40, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.5, reg_lambda=5, random_state=seed,
                                  verbosity=-1),
        "xgb": xgb.XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=4,
                                 min_child_weight=3, subsample=0.8, colsample_bytree=0.5,
                                 reg_lambda=5, tree_method="hist", random_state=seed,
                                 eval_metric="logloss", n_jobs=-1),
        "cat": CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=5,
                                  l2_leaf_reg=8, random_seed=seed, verbose=False,
                                  allow_writing_files=False, thread_count=-1),
    }


def run_clf(use_w):
    tag = "w" if use_w else "nw"
    oof = {k: np.zeros(len(X)) for k in ["lgb", "xgb", "cat"]}
    tst = {k: np.zeros(len(Xt)) for k in oof}
    for seed in SEEDS:
        skf = StratifiedKFold(NF, shuffle=True, random_state=seed)
        for f, (a, b) in enumerate(skf.split(X, y)):
            for name, m in clf_models(seed).items():
                t = time.time()
                m.fit(X.iloc[a], y[a], sample_weight=w[a] if use_w else None)
                oof[name][b] += m.predict_proba(X.iloc[b])[:, 1] / len(SEEDS)
                tst[name] += m.predict_proba(Xt)[:, 1] / (len(SEEDS) * NF)
            print(f"[{tag}] seed {seed} fold {f} done", flush=True)
    for k in oof:
        np.save(f"oof/p_{k}_{tag}.npy", oof[k]); np.save(f"oof/pt_{k}_{tag}.npy", tst[k])
        print(tag, k, "AUC %.4f AP %.4f LL %.5f" % (roc_auc_score(y, oof[k]),
              average_precision_score(y, oof[k]), log_loss(y, oof[k])))
    avg = np.mean([oof[k] for k in oof], 0)
    print(tag, "avg AUC %.4f AP %.4f" % (roc_auc_score(y, avg), average_precision_score(y, avg)))


def run_cost():
    pos = np.where(y == 1)[0]
    Xp, lc = X.iloc[pos], np.log(cost[pos])
    oof = np.zeros(len(pos)); tst_all = np.zeros(len(Xt)); trn_all = np.zeros(len(X))
    for seed in SEEDS:
        for a, b in KFold(NF, shuffle=True, random_state=seed).split(Xp):
            ms = [lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=7,
                                    min_child_samples=10, subsample=0.8, subsample_freq=1,
                                    colsample_bytree=0.5, reg_lambda=10, random_state=seed,
                                    verbosity=-1),
                  CatBoostRegressor(iterations=600, learning_rate=0.03, depth=4, l2_leaf_reg=10,
                                    random_seed=seed, verbose=False, allow_writing_files=False)]
            for m in ms:
                m.fit(Xp.iloc[a], lc[a])
                oof[b] += m.predict(Xp.iloc[b]) / (2 * len(SEEDS))
                tst_all += m.predict(Xt) / (2 * len(SEEDS) * NF)
                trn_all += m.predict(X) / (2 * len(SEEDS) * NF)
    # For non-failed train rows use the average of fold models (they never saw them).
    trn_all[pos] = oof
    np.save("oof/mu_train.npy", trn_all); np.save("oof/mu_test.npy", tst_all)
    r = np.corrcoef(oof, lc)[0, 1]
    print("cost model: corr(oof, log cost)=%.3f, RMSE %.3f vs const %.3f" % (
        r, np.sqrt(np.mean((oof - lc) ** 2)), lc.std()))


if __name__ == "__main__":
    what = sys.argv[1:] or ["cost", "w", "nw"]
    if "cost" in what: run_cost()
    if "w" in what: run_clf(True)
    if "nw" in what: run_clf(False)

In [ ]:
%run oof_models.py cost w

### Step 2: `full_fit.py`
Fit the ensemble on all training data; predict test both as-is and after quantile-matching each test sensor to the training distribution (removes the remaining depot-level scale shift in the heavy-usage tail).

In [ ]:
%%writefile full_fit.py
"""Fit the ensemble on all training data; predict test both as-is and after
quantile-matching each test sensor to the training distribution (removes the
remaining depot-level scale shift in the heavy-usage tail)."""
import numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from fleet_prep import load, add_features

def qmatch(train_col, test_col):
    a = np.sort(train_col.dropna().to_numpy()); t = test_col.copy()
    m = t.notna()
    r = t[m].rank(pct=True, method="average").to_numpy()
    t[m] = np.quantile(a, np.clip(r - 0.5 / m.sum(), 0, 1))
    return t

tr, te, X, Xt = load()
sensors = [c for c in Xt.columns if c.startswith("s") and "_" not in c]
te_q = te.copy()
for c in sensors:
    te_q[c] = qmatch(tr[c], te[c])
Xq = add_features(te_q, sensors)
y = tr.failure.to_numpy(); cost = tr.repair_cost.fillna(0).to_numpy()
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)
P, PQ = [], []
for seed in [0, 1]:
    ms = [lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=40,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=5,
                             random_state=seed, verbosity=-1),
          xgb.XGBClassifier(n_estimators=600, learning_rate=0.02, max_depth=4, min_child_weight=3,
                            subsample=0.8, colsample_bytree=0.5, reg_lambda=5, tree_method="hist",
                            random_state=seed, n_jobs=-1),
          CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=5, l2_leaf_reg=8,
                             random_seed=seed, verbose=False, allow_writing_files=False)]
    for m in ms:
        m.fit(X, y, sample_weight=w)
        P.append(m.predict_proba(Xt)[:, 1]); PQ.append(m.predict_proba(Xq)[:, 1])
pos = y == 1
MU, MUQ = [], []
for seed in [0, 1]:
    for m in [lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=7, min_child_samples=10,
                                subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=10,
                                random_state=seed, verbosity=-1),
              CatBoostRegressor(iterations=600, learning_rate=0.03, depth=4, l2_leaf_reg=10,
                                random_seed=seed, verbose=False, allow_writing_files=False)]:
        m.fit(X[pos], np.log(cost[pos])); MU.append(m.predict(Xt)); MUQ.append(m.predict(Xq))
np.save("oof/full_pt.npy", np.mean(P, 0)); np.save("oof/full_ptq.npy", np.mean(PQ, 0))
np.save("oof/full_mut.npy", np.mean(MU, 0)); np.save("oof/full_mutq.npy", np.mean(MUQ, 0))
print("mean test p: as-is %.4f  quantile-matched %.4f  (train rate %.4f)" % (np.mean(P), np.mean(PQ), y.mean()))

In [ ]:
%run full_fit.py

### Step 3: `make_risk_variant.py`
Write a submission from the full-data ensemble with a given test-depot risk multiplier. Usage: python make_risk_variant.py <alpha> <name> [budget_mult]

In [ ]:
%%writefile make_risk_variant.py
"""Write a submission from the full-data ensemble with a given test-depot risk multiplier.
Usage: python make_risk_variant.py <alpha> <name> [budget_mult]"""
import sys
import numpy as np, pandas as pd
from policy import decide

alpha, name = float(sys.argv[1]), sys.argv[2]
tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost"])
te = pd.read_csv("data/test.csv", usecols=["id"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
mu = np.load("oof/mu_train.npy"); eps = np.log(c[y == 1]) - mu[y == 1]
bm = float(sys.argv[3]) if len(sys.argv) > 3 else 1.0
p = np.load("oof/full_pt.npy")
f, b = decide(p, np.load("oof/full_mut.npy"), eps, alpha, 1.0, 0.0, bm)
sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
assert len(sub) == 20000 and sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all()
sub.to_csv(f"submission_{name}.csv", index=False)
print(name, "alpha", alpha, "budget_mult", bm, "inspections", int(f.sum()))

In [ ]:
%run make_risk_variant.py 9 v24_risk9

### Step 4: `make_risk_variant.py 12 v25_risk12`

In [ ]:
%run make_risk_variant.py 12 v25_risk12

## Output files

In [ ]:
import glob, pandas as pd
for p in sorted(glob.glob("submission_*.csv")):
    s = pd.read_csv(p)
    print(f"{p}: {int(s.flag.sum())} inspections, mean budget {s.budget[s.flag == 1].mean():.0f}")

## Other scripts in this version
Experiments and tuning scripts that are not needed for this version's submission. Shown for completeness; they are written to disk but not run.

### `logit_model.py`
Logistic regression on log1p sensors. Unlike trees it extrapolates, which matters because the test depot's usage tail goes 1.3-1.8x beyond the training range.

In [ ]:
%%writefile logit_model.py
"""Logistic regression on log1p sensors. Unlike trees it extrapolates, which matters
because the test depot's usage tail goes 1.3-1.8x beyond the training range."""
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from fleet_prep import load

tr, te, X, Xt = load()
y = tr.failure.to_numpy()
w = np.where((y == 0) & (tr.followup_days < 365), tr.followup_days / 365.0, 1.0)
cols = [c for c in X.columns if not c.endswith("_frac")]
sl = lambda a: np.sign(a) * np.log1p(np.abs(a))

def model(C):
    return make_pipeline(FunctionTransformer(sl), SimpleImputer(strategy="median", add_indicator=True),
                         StandardScaler(), LogisticRegression(C=C, max_iter=5000))

for C in [0.003, 0.01, 0.03, 0.1]:
    oof = np.zeros(len(X))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
        m = model(C).fit(X[cols].iloc[a], y[a], logisticregression__sample_weight=w[a])
        oof[b] = m.predict_proba(X[cols].iloc[b])[:, 1]
    print(f"C={C}: AUC {roc_auc_score(y, oof):.4f} AP {average_precision_score(y, oof):.4f}")
    np.save(f"oof/p_lr_C{C}.npy", oof)
    m = model(C).fit(X[cols], y, logisticregression__sample_weight=w)
    np.save(f"oof/pt_lr_C{C}.npy", m.predict_proba(Xt[cols])[:, 1])

### `make_grid.py`
v16-v23: one-knob variants around v14, for leaderboard comparison of global settings.

In [ ]:
%%writefile make_grid.py
"""v16-v23: one-knob variants around v14, for leaderboard comparison of global settings.

Each file changes exactly ONE setting relative to v14 (1692 inspections, EM prior,
heteroscedastic budgets, bagged probabilities), so a leaderboard score tells us
which direction that setting should move. No per-vehicle label probing: every
change is a single global parameter applied to all vehicles.

v16  budgets x0.8               (CV-optimal lower end)
v17  budgets x1.15              (slightly larger budgets)
v18  fewer inspections: 1450    (only the riskiest vehicles)
v19  more inspections: 1950     (cast a wider net)
v20  homoscedastic budgets      (v12-style budgets on v14 flags: tests the sigma model)
v21  v9's flags + v14 budgets   (isolates the budget model vs v9)
v22  (dropped: 1685/1692 flags identical to v14, not worth a submission)
v23  budgets x0.9 and 1800 inspections (combined mild move)
"""
import numpy as np, pandas as pd
from policy import decide
from cv_eval import y, cost
from make_v12_v14 import em_adjust, write, q14, f14, mut, sig_t, z, eps, ALPHA

base_b = decide(q14, mut, z, ALPHA, 1.0, -1e9, sigma=sig_t)[1]   # hetero budget for every vehicle
homo_b = decide(q14, mut, eps, ALPHA, 1.0, -1e9)[1]


def flags_for(n):
    """Same policy as v14, alpha scaled so exactly ~n vehicles are flagged."""
    lo, hi = 0.05, 50.0
    for _ in range(40):
        mid = np.sqrt(lo * hi)
        k = decide(q14, mut, eps, ALPHA * mid, 1.0, 0.0)[0].sum()
        lo, hi = (mid, hi) if k < n else (lo, mid)
    return decide(q14, mut, eps, ALPHA * hi, 1.0, 0.0)[0]


def keep(f, b):
    return np.where(f == 1, b, 0.0)


write("v16_budget080", f14, keep(f14, base_b * 0.8))
write("v17_budget115", f14, keep(f14, base_b * 1.15))
f18 = flags_for(1450); write("v18_insp1450", f18, keep(f18, base_b))
f19 = flags_for(1950); write("v19_insp1950", f19, keep(f19, base_b))
write("v20_homo_budget", f14, keep(f14, homo_b))
f9 = pd.read_csv("submission_v9_risk6.csv").flag.to_numpy()
write("v21_v9flags_v14budget", f9, keep(f9, base_b))
score = q14 * np.exp(mut + 0.5 * np.var(eps))
f22 = np.zeros(len(q14), int); f22[np.argsort(-score)[:int(f14.sum())]] = 1
# v22 not written: nearly identical to v14
f23 = flags_for(1800); write("v23_budget090_insp1800", f23, keep(f23, base_b * 0.9))

for name, f in [("v18", f18), ("v19", f19), ("v21", f9), ("v22", f22), ("v23", f23)]:
    print(f"{name}: overlap with v14 flags {int((f & f14).sum())}/{int(f14.sum())}")

### `make_submission.py`
Build the final submission from saved ensemble predictions (run oof_models.py first).

In [ ]:
%%writefile make_submission.py
"""Build the final submission from saved ensemble predictions (run oof_models.py first)."""
import numpy as np, pandas as pd
from policy import decide

ALPHA, EPS_SCALE, MARGIN = 1.5, 1.0, 2.0   # chosen on out-of-fold cost (0.838 vs 3.49 do-nothing)
tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost"])
te = pd.read_csv("data/test.csv", usecols=["id"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
mu = np.load("oof/mu_train.npy"); mut = np.load("oof/mu_test.npy")
eps = np.log(c[y == 1]) - mu[y == 1]
pt = np.mean([np.load(f"oof/pt_{n}_w.npy") for n in ("lgb", "xgb", "cat")], 0)
f, b = decide(pt, mut, eps, ALPHA, EPS_SCALE, MARGIN)
sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
samp = pd.read_csv("data/sample_submission.csv")
assert list(sub.columns) == list(samp.columns) and (sub.id.values == samp.id.values).all()
assert sub.flag.isin([0, 1]).all() and np.isfinite(sub.budget).all() and (sub.budget >= 0).all()
assert (sub.loc[sub.flag == 0, "budget"] == 0).all()
sub.to_csv("submission_v5_ensemble.csv", index=False)
print("flagged", sub.flag.sum(), "of", len(sub), "| median budget", sub.loc[sub.flag == 1, "budget"].median())

### `make_variants.py`
Write submission variants from the full-data fit (run full_fit.py first).

In [ ]:
%%writefile make_variants.py
"""Write submission variants from the full-data fit (run full_fit.py first)."""
import numpy as np, pandas as pd
from policy import decide

tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost"])
te = pd.read_csv("data/test.csv", usecols=["id"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
mu = np.load("oof/mu_train.npy"); eps = np.log(c[y == 1]) - mu[y == 1]
V = {
    # quantile-matched test sensors, same policy settings as v5
    "v6_qmatch": (np.load("oof/full_ptq.npy"), np.load("oof/full_mutq.npy"), 1.5, 1.0, 2.0),
    # v5 inputs but more inspections (use if v6 scores worse than v5)
    "v7_aggressive": (np.load("oof/full_pt.npy"), np.load("oof/full_mut.npy"), 2.5, 1.0, 0.0),
}
for name, (p, m, a, s, mg) in V.items():
    f, b = decide(p, m, eps, a, s, mg)
    sub = pd.DataFrame({"id": te.id, "flag": f.astype(int), "budget": np.round(b, 2)})
    assert sub.flag.isin([0, 1]).all() and (sub.budget >= 0).all() and len(sub) == 20000
    sub.to_csv(f"submission_{name}.csv", index=False)
    fl = sub.flag == 1
    print(f"{name}: flags={fl.sum()}  sum(10+0.02*budget)={10*fl.sum()+0.02*sub.budget[fl].sum():.0f}")

### `pseudo_test.py`
Pseudo-test validation: reweight training vehicles by the train->test density ratio (domain classifier odds) and score policies on the reweighted OOF predictions.

In [ ]:
%%writefile pseudo_test.py
"""Pseudo-test validation: reweight training vehicles by the train->test density ratio
(domain classifier odds) and score policies on the reweighted OOF predictions.

Result: reweighted failure rate is 1.9% (EM estimate for test: 3.5%), and the
reweighted cost prefers risk x1.5 while the leaderboard improved steadily up to x6.
The test depot's extra risk is mostly outside the training range, so covariate
reweighting cannot reproduce it; this proxy is not used to pick the risk level.
It does agree with CV that budgets x0.8-1.0 are best.
"""
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from fleet_prep import load
from policy import decide, competition_cost
from cv_eval import y, cost, weights

tr, te, X, Xt = load()
A = pd.concat([X, Xt]); d = np.r_[np.zeros(len(X)), np.ones(len(Xt))]
m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=100,
                       colsample_bytree=0.5, subsample=0.8, subsample_freq=1, verbosity=-1)
pd_ = cross_val_predict(m, A, d, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
pt = np.clip(pd_[:len(X)], 1e-4, 1 - 1e-4)
r = np.clip(pt / (1 - pt) * len(X) / len(Xt), 0, None); r = np.minimum(r, np.quantile(r, 0.999))
print("reweighted failure rate %.4f" % np.average(y, weights=weights * r))
mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
p = np.mean([np.load(f"oof/p_{n}_w.npy") for n in ("lgb", "xgb", "cat")], 0)
for a in [1.5, 2.5, 4, 6, 8]:
    cc = competition_cost(cost, *decide(p, mu, eps, a, 1.0, 0))
    print(f"alpha {a}: train {cc.mean():.4f}  pseudo-test {np.average(cc, weights=r):.4f}")

### `sigma_model.py`
Heteroscedastic repair-cost spread: predict |log-cost residual| per vehicle.

In [ ]:
%%writefile sigma_model.py
"""Heteroscedastic repair-cost spread: predict |log-cost residual| per vehicle.

Saves oof/sigma_train.npy (OOF for failed vehicles, fold-average elsewhere),
oof/sigma_test.npy and oof/z.npy (standardised residuals of failed vehicles).
"""
import numpy as np, lightgbm as lgb
from sklearn.model_selection import KFold
from fleet_prep import load
from cv_eval import y, cost

_, _, X, Xt = load()
pos = np.where(y == 1)[0]
mu = np.load("oof/mu_train.npy")
res = np.log(cost[pos]) - mu[pos]
Xp = X.iloc[pos]
oof = np.zeros(len(pos)); s_all = np.zeros(len(X)); s_test = np.zeros(len(Xt))
SEEDS, NF = [0, 1, 2], 5
for seed in SEEDS:
    for a, b in KFold(NF, shuffle=True, random_state=seed).split(Xp):
        m = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=4, min_child_samples=15,
                              random_state=seed, verbosity=-1).fit(Xp.iloc[a], np.abs(res[a]))
        oof[b] += m.predict(Xp.iloc[b]) / len(SEEDS)
        s_all += m.predict(X) / (len(SEEDS) * NF)
        s_test += m.predict(Xt) / (len(SEEDS) * NF)
s_all[pos] = oof
s_all, s_test = np.maximum(s_all, 0.05), np.maximum(s_test, 0.05)
np.save("oof/sigma_train.npy", s_all); np.save("oof/sigma_test.npy", s_test)
np.save("oof/z.npy", res / s_all[pos])
print("sigma train mean %.3f  test mean %.3f" % (s_all.mean(), s_test.mean()))

### `tune.py`
Tune the decision policy on out-of-fold predictions and compare to baselines.

In [ ]:
%%writefile tune.py
"""Tune the decision policy on out-of-fold predictions and compare to baselines."""
import itertools, sys
import numpy as np
import pandas as pd
from policy import decide, competition_cost

tr = pd.read_csv("data/train.csv", usecols=["failure", "repair_cost", "followup_days"])
y = tr.failure.to_numpy(); c = tr.repair_cost.fillna(0).to_numpy()
pos = y == 1
mu = np.load("oof/mu_train.npy")
eps = np.log(c[pos]) - mu[pos]


def P(tag, names=("lgb", "xgb", "cat")):
    return np.mean([np.load(f"oof/p_{n}_{tag}.npy") for n in names], 0)


def score(flag, b):
    return competition_cost(c, flag, b).mean()


print("Do nothing (never inspect):          %.4f" % c.mean())
q90 = np.quantile(c[pos], 0.9)
for tag in ["w", "nw"]:
    for names in [("lgb",), ("xgb",), ("cat",), ("lgb", "xgb", "cat")]:
        p = P(tag, names)
        # notebook-style baseline: threshold on p*exp(mu), fixed q90 budget
        e = p * np.exp(mu)
        best = min(score((e > t).astype(int), np.where(e > t, q90, 0))
                   for t in np.quantile(e, np.linspace(0.9, 0.9995, 200)))
        f, b = decide(p, mu, eps)
        print(f"{tag:>2} {'+'.join(names):12s} threshold+q90 best={best:.4f} | "
              f"optimal policy (no tuning)={score(f, b):.4f} flags={f.sum()}")

tag = sys.argv[1] if len(sys.argv) > 1 else "w"
p = P(tag)
rows = []
for a, s, m, bm in itertools.product([0.6, 0.8, 1.0, 1.2, 1.5], [0.8, 1.0, 1.2],
                                     [-2, 0, 2, 5], [0.8, 1.0, 1.2]):
    f, b = decide(p, mu, eps, a, s, m, bm)
    rows.append((a, s, m, bm, score(f, b), f.sum()))
r = pd.DataFrame(rows, columns=["alpha", "eps_scale", "margin", "budget_mult", "cost", "flags"])
print(r.sort_values("cost").head(10).to_string(index=False))

### `tune_budget.py`
Compare budget models on failed vehicles via CV pinball loss at tau=0.909 (the competition's under/over penalty ratio 0.20/0.22).

In [ ]:
%%writefile tune_budget.py
"""Compare budget models on failed vehicles via CV pinball loss at tau=0.909
(the competition's under/over penalty ratio 0.20/0.22)."""
import numpy as np, lightgbm as lgb
from sklearn.model_selection import KFold
from fleet_prep import load
from cv_eval import y, cost
TAU = 0.2 / 0.22
_, _, X, _ = load()
pos = np.where(y == 1)[0]; Xp = X.iloc[pos]; C = cost[pos]; lc = np.log(C)
mu = np.load("oof/mu_train.npy")[pos]
def pin(b): return np.mean(0.2 * np.maximum(C - b, 0) + 0.02 * np.maximum(b - C, 0))
print("global quantile budget           %.2f" % pin(np.quantile(C, TAU)))
# current: exp(mu + quantile of OOF residuals)
res = lc - mu
print("current mu + residual quantile   %.2f" % pin(np.exp(mu + np.quantile(res, TAU))))
for s in [0.8, 1.2]:
    print(f"  residual scale {s}              %.2f" % pin(np.exp(mu + s * np.quantile(res, TAU))))
# direct quantile regression on log cost, with OOF
for leaves, n in [(4, 200), (7, 300), (7, 600)]:
    oof = np.zeros(len(pos))
    for seed in [0, 1, 2]:
        for a, b in KFold(5, shuffle=True, random_state=seed).split(Xp):
            m = lgb.LGBMRegressor(objective="quantile", alpha=TAU, n_estimators=n, learning_rate=0.03,
                                  num_leaves=leaves, min_child_samples=10, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.5, random_state=seed, verbosity=-1)
            m.fit(Xp.iloc[a], lc[a]); oof[b] += m.predict(Xp.iloc[b]) / 3
    print(f"LGBM quantile on log cost ({leaves} leaves, {n} trees) %.2f" % pin(np.exp(oof)))
# heteroscedastic: model |residual| and scale
oof_s = np.zeros(len(pos))
for seed in [0, 1, 2]:
    for a, b in KFold(5, shuffle=True, random_state=seed).split(Xp):
        m = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=4, min_child_samples=15,
                              random_state=seed, verbosity=-1).fit(Xp.iloc[a], np.abs(res[a]))
        oof_s[b] += m.predict(Xp.iloc[b]) / 3
z = res / np.maximum(oof_s, 0.05)
print("heteroscedastic mu + s(x)*q(z)    %.2f" % pin(np.exp(mu + np.maximum(oof_s, 0.05) * np.quantile(z, TAU))))

### `tune_lgb.py`
Small CV search over LightGBM settings (same folds for every config).

In [ ]:
%%writefile tune_lgb.py
"""Small CV search over LightGBM settings (same folds for every config)."""
import numpy as np, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from fleet_prep import load
from cv_eval import y, cost, weights, report

_, _, X, _ = load()
mu = np.load("oof/mu_train.npy"); eps = np.log(cost[y == 1]) - mu[y == 1]
base = dict(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=40, subsample=0.8,
            subsample_freq=1, colsample_bytree=0.5, reg_lambda=5)
grid = {
    "base": {},
    "leaves7": dict(num_leaves=7),
    "leaves31": dict(num_leaves=31, min_child_samples=20),
    "trees1200_lr01": dict(n_estimators=1200, learning_rate=0.01),
    "col0.3": dict(colsample_bytree=0.3),
    "minchild80": dict(min_child_samples=80),
    "lambda20": dict(reg_lambda=20),
    "trees300": dict(n_estimators=300),
    "trees1000": dict(n_estimators=1000),
    "pos_w3": dict(scale_pos_weight=3),
}
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(X, y))
for name, kw in grid.items():
    oof = np.zeros(len(X))
    for a, b in folds:
        m = lgb.LGBMClassifier(**{**base, **kw}, random_state=0, verbosity=-1)
        m.fit(X.iloc[a], y[a], sample_weight=weights[a])
        oof[b] = m.predict_proba(X.iloc[b])[:, 1]
    report(name, oof, mu, eps)